In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

import sys
sys.path.append(os.path.abspath(".."))
from function import DOWN_raw

from playsound import playsound

import warnings
warnings.filterwarnings('ignore')

playsound is relying on another python subprocess. Please use `pip install pygobject` if you want playsound to run more efficiently.


## Export the RE and important info from Corrected RSR/Ensemble data

In [2]:
ENSEMBLE_SAT = 'ALL'
# product, time_reso = 'ENSEMBLE_mean', '1dy'
# product, time_reso = 'ENSEMBLE_median', '1dy'

product, time_reso = 'IMERG', '1dy'
# product, time_reso = 'CMORPH', '3h'
# product, time_reso = 'ERA5', '3h'
# product, time_reso = 'MSWEP', '3h'
# product, time_reso = 'CHIRPS', '1dy'
# product, time_reso = 'GSMaP', '3h'

In [3]:
# correction, nameout, label = 'Log-Linear-Regression', 'LLC', 'Log-Linear Regression'
# correction, nameout, label = 'Linear-Regression', 'LLR', 'Linear Regression'
correction, nameout, label = 'Linear-Regression-origin', 'LTO', 'Linear Regression Origin'

# correction, nameout, label = 'Quantile Delta Mapping', 'QDM', 'Quantile Delta Mapping'
# correction, nameout, label = 'Cummulative Distribution', 'CDFt', 'Cummulative Distribution'
# correction, nameout, label = 'Quantile-Quantile', 'QQc', 'Quantile Quantile Mapping'

In [4]:
frac = 0.7

In [5]:
lon_min, lon_max, lat_min, lat_max, area, toll = 6.5, 19, 36.5, 48, 'ITALY', 0.002

Tr = [5,  10,  20,  50, 100, 200]
Fi = 1 - 1/np.array(Tr)

veneto_dir = os.path.join('/','media','arturo','T9','Data','shapes','Europa','Italy')

if os.path.exists(veneto_dir):
    REGIONS = gpd.read_file(os.path.join(veneto_dir,'Italy_regions.geojson'))
else:
    raise SystemExit(f"File not found: {veneto_dir}")

obs_base = os.path.join('/','media','arturo','T9','Data','Italy','Rain_Gauges_QC')
bias_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','PARAM')
ouput_base = os.path.join('/','media','arturo','T9','Data','Italy','statistics','PARAM')

METADATA = pd.read_csv(os.path.join(obs_base, 'data', 'METADATA', 'METADATA_FTS_QCv4_Case1_wAIRHO_v3_1dy.csv'))
METADATA["Lat"] = np.round(METADATA["Lat"], 6)
METADATA["Lon"] = np.round(METADATA["Lon"], 6)

In [6]:
# seeds_list = [7, 19, 31, 53, 89, 127, 211, 307, 401, 509, 613, 727, 839, 947, 1051]
seeds_list = [7, 19, 31, 53, 89, 127, 211, 307]
# seeds_list = [7]

for seed in seeds_list:
    print(f'Seed: {seed}')

    ISO_list = METADATA.ISO.unique()

    Q_train_list = []
    Q_val_list = []

    for iso in METADATA['ISO'].unique():
        
        META_iso = METADATA[METADATA['ISO'] == iso]

        # Si una región tiene muy pocas estaciones, evita errores
        if len(META_iso) < 2:
            Q_train_list.append(META_iso)
            continue

        META_80 = META_iso.sample(frac=frac, random_state=seed)
        META_20 = META_iso.drop(META_80.index)

        Q_val_list.append(META_20)

    Q_val = pd.concat(Q_val_list, ignore_index=True)

    # Base down
    if product == 'ENSEMBLE_mean':
        dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','5_ENSEMBLE')
        dir_input = os.path.join(os.path.join(dir_base, 'ITALY_ENSEMBLE_ALL_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_mean.nc'))
    elif product == 'ENSEMBLE_median':
        dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','5_ENSEMBLE')
        dir_input = os.path.join(os.path.join(dir_base, 'ITALY_ENSEMBLE_ALL_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_median.nc'))
    else:
        dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','5_DOWN')
        dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson.nc'))
    data_down = xr.open_dataset(dir_input)
    
    # Correction
    dir_LLc = os.path.join(bias_base, 'LLc', f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_LLc_{str(seed).zfill(4)}.nc')
    data_LLc = xr.open_dataset(dir_LLc)
    
    dir_LTO = os.path.join(bias_base, 'LTO', f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_LTO_{str(seed).zfill(4)}.nc')
    data_LTO = xr.open_dataset(dir_LTO)
    
    dir_QQc = os.path.join(bias_base, 'QQc', f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_{str(seed).zfill(4)}.nc')
    data_QQc = xr.open_dataset(dir_QQc)
    
    dir_QDM = os.path.join(bias_base, 'QDM', f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QDM_{str(seed).zfill(4)}.nc')
    data_QDM = xr.open_dataset(dir_QDM)

    Tr_index = 3

    ISO_names = np.unique(METADATA.ISO.values)

    INFO_region = {}
    WEIBULL_region = {}
    QUANTILES_region = {}

    for rr in range(len(ISO_names)):
        region_ISO = ISO_names[rr]

        INFO_dict = {}
        WEIBULL_dict = {}
        QUANTILES_dict = {}

        # print(f'{rr+1}: {region_ISO}')

        METADATA_clear = Q_val[Q_val['ISO']==region_ISO].reset_index(inplace=False)

        for nn in range(len(METADATA_clear)):#len(METADATA_clear)
            filename = f'{METADATA_clear['File'].values[nn]}'
            lat_obs = METADATA_clear['Lat'][nn]
            lon_obs = METADATA_clear['Lon'][nn]
            elev_obs = METADATA_clear['DEM_Elevation'][nn]

            OBS_pd = pd.read_csv(os.path.join(obs_base, 'Weibull', '1dy', region_ISO, filename))
            OBS_pd = OBS_pd[(OBS_pd['Year']>=2002)&(OBS_pd['Year']<=2023)].reset_index(drop=True)
            
            if len(OBS_pd) == 0:
                    continue

            else:
                OBS_N = OBS_pd['N'].values
                OBS_C = OBS_pd['C'].values
                OBS_W = OBS_pd['W'].values
                OBS_Y = OBS_pd['Year'].values

                mask = ~np.isnan(OBS_N)

                OBS_N = OBS_N[mask]
                OBS_C = OBS_C[mask]
                OBS_W = OBS_W[mask]
                OBS_Y = OBS_Y[mask]

                if len(OBS_Y) >= 8: # greather than 8 years

                    x0 = np.nanmean(OBS_C)
                    OBS_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, OBS_N, OBS_C, OBS_W, thresh=1)
                    OBS_Q2 = np.where(flag, OBS_Q, np.nan)
                    
                    # =============================================================================================================================================
                    PREC_DOWN = data_down.sel(lat=lat_obs, lon=lon_obs, method='nearest')
                    
                    Sat_down = pd.DataFrame({'Year':PREC_DOWN.year.values, 'N':PREC_DOWN.NYd.values, 'C':PREC_DOWN.CYd.values, 'W':PREC_DOWN.WYd.values})
                    Sat_down = Sat_down.set_index('Year').loc[OBS_pd['Year']].reset_index()
                    x0 = np.nanmean(Sat_down.C.values)
                    SAT_down_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, Sat_down.N.values, Sat_down.C.values, Sat_down.W.values, thresh=1)
                    Mevd = np.where(flag, SAT_down_Q, np.nan)

                    # =============================================================================================================================================
                    PREC_LTO = data_LTO.sel(lat=lat_obs, lon=lon_obs, method='nearest')
                    lat_ref = float(PREC_LTO.lat.values)
                    lon_ref = float(PREC_LTO.lon.values)

                    INFO = pd.DataFrame({'lat_obs':[lat_obs], 'lon_obs':[lon_obs], 'elev_obs':[elev_obs], 'lat_ref':[lat_ref], 'lon_ref':[lon_ref]})
                    INFO_dict[filename] = INFO

                    Sat_down_LTO = pd.DataFrame({'Year':PREC_LTO.year.values, 'N':PREC_LTO.NYd.values, 'C':PREC_LTO.CYd.values, 'W':PREC_LTO.WYd.values})
                    Sat_down_LTO = Sat_down_LTO.set_index('Year').loc[OBS_pd['Year']].reset_index()
                    x0 = np.nanmean(Sat_down_LTO.C.values)
                    SAT_down_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, Sat_down_LTO.N.values, Sat_down_LTO.C.values, Sat_down_LTO.W.values, thresh=1)
                    Mevd_LTO = np.where(flag, SAT_down_Q, np.nan)

                    # =============================================================================================================================================
                    PREC_LLc = data_LLc.sel(lat=lat_obs, lon=lon_obs, method='nearest')
                    
                    Sat_down_LLc = pd.DataFrame({'Year':PREC_LLc.year.values, 'N':PREC_LLc.NYd.values, 'C':PREC_LLc.CYd.values, 'W':PREC_LLc.WYd.values})
                    Sat_down_LLc = Sat_down_LLc.set_index('Year').loc[OBS_pd['Year']].reset_index()
                    x0 = np.nanmean(Sat_down_LLc.C.values)
                    SAT_down_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, Sat_down_LLc.N.values, Sat_down_LLc.C.values, Sat_down_LLc.W.values, thresh=1)
                    Mevd_LLc = np.where(flag, SAT_down_Q, np.nan)
                    
                    # =============================================================================================================================================
                    PREC_QQc = data_QQc.sel(lat=lat_obs, lon=lon_obs, method='nearest')
                    
                    Sat_down_QQc = pd.DataFrame({'Year':PREC_QQc.year.values, 'N':PREC_QQc.NYd.values, 'C':PREC_QQc.CYd.values, 'W':PREC_QQc.WYd.values})
                    Sat_down_QQc = Sat_down_QQc.set_index('Year').loc[OBS_pd['Year']].reset_index()
                    x0 = np.nanmean(Sat_down_QQc.C.values)
                    SAT_down_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, Sat_down_QQc.N.values, Sat_down_QQc.C.values, Sat_down_QQc.W.values, thresh=1)
                    Mevd_QQc = np.where(flag, SAT_down_Q, np.nan)
                    
                    # =============================================================================================================================================
                    PREC_QDM = data_QDM.sel(lat=lat_obs, lon=lon_obs, method='nearest')
                    
                    Sat_down_QDM = pd.DataFrame({'Year':PREC_QDM.year.values, 'N':PREC_QDM.NYd.values, 'C':PREC_QDM.CYd.values, 'W':PREC_QDM.WYd.values})
                    Sat_down_QDM = Sat_down_QDM.set_index('Year').loc[OBS_pd['Year']].reset_index()
                    x0 = np.nanmean(Sat_down_QDM.C.values)
                    SAT_down_Q, flag = DOWN_raw.mev_quant_update(Fi, x0, Sat_down_QDM.N.values, Sat_down_QDM.C.values, Sat_down_QDM.W.values, thresh=1)
                    Mevd_QDM = np.where(flag, SAT_down_Q, np.nan)

                    QUANTILES = pd.DataFrame({'Tr':Tr, 'OBS':OBS_Q2, 'DOWN':Mevd, 'LLc':Mevd_LLc, 'LTO':Mevd_LTO, 'QQc':Mevd_QQc, 'LSc':Mevd_QDM})

                    
                    # WEIBULL_dict[filename] = WEIBULL
                    QUANTILES_dict[filename] = QUANTILES

        INFO_region[region_ISO] = INFO_dict
        # WEIBULL_region[region_ISO] = WEIBULL_dict
        QUANTILES_region[region_ISO] = QUANTILES_dict

    dir_out = os.path.join('/','media','arturo','T9','Data','Italy','statistics','PARAM')
    hdf5_file = os.path.join(dir_out, f'statistics_obs_{product}_corrected_QQc_LLc_{str(seed).zfill(4)}.h5')

    print(f'Export as: {hdf5_file}')

    with pd.HDFStore(hdf5_file, mode='w') as store:

        for region_ISO in INFO_region.keys():
            stations = INFO_region[region_ISO].keys()  # las estaciones de la región

            for station in stations:

                info_df = INFO_region[region_ISO][station]
                # weibull_df = WEIBULL_region[region_ISO][station]
                quantiles_df = QUANTILES_region[region_ISO][station]

                store[f"/{region_ISO}/{station}/INFO"] = info_df
                # store[f"/{region_ISO}/{station}/WEIBULL"] = weibull_df
                store[f"/{region_ISO}/{station}/QUANTILES"] = quantiles_df
    print()

Seed: 7
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0007.h5

Seed: 19
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0019.h5

Seed: 31
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0031.h5

Seed: 53
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0053.h5

Seed: 89
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0089.h5

Seed: 127
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0127.h5

Seed: 211
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0211.h5

Seed: 307
Export as: /media/arturo/T9/Data/Italy/statistics/PARAM/statistics_obs_IMERG_corrected_QQc_LLc_0307.h5



In [7]:
playsound("../sound/HOMER_DOH.mp3")